# PHASE - ESC-50 dry run

Closes the last Phase 3 exit criterion: the supervised baseline must train end to end
and reach a sane accuracy.

This is a **plumbing test, not a result**. It exists so that when DeepShip training
misbehaves, the pipeline is not a suspect.

**Expected: 60-80% accuracy** for ResNet-18 on ESC-50 log-Mel. Section 7 prints an
explicit verdict, fixed in advance so it cannot be rationalised after the fact.

> Upload this notebook directly (File -> Upload notebook). Do **not** open it from
> GitHub - the pushed copy is behind.

Runtime -> Change runtime type -> **T4 GPU** first.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1. Repository and dependencies


In [ ]:
import os, pathlib

REPO = pathlib.Path('/content/deepsonar_v2')
if not REPO.exists():
    !git clone --depth 1 https://github.com/Adhi-1004/deepsonar_v2.git {REPO}
os.chdir(REPO)
print(pathlib.Path.cwd())


In [ ]:
%pip install -q -e '.[data]' 2>&1 | tail -3

import importlib.util
import torch, phase

print('phase', phase.__version__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print()
for module in ('kaggle', 'librosa', 'soundfile', 'wandb', 'sklearn'):
    found = importlib.util.find_spec(module) is not None
    print(module.ljust(12), 'installed' if found else 'MISSING')


## 2. Kaggle credentials

Upload `kaggle.json` when prompted, or set `KAGGLE_USERNAME` and `KAGGLE_KEY` first.


In [ ]:
import pathlib

cred = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
if not cred.exists() and not os.environ.get('KAGGLE_KEY'):
    from google.colab import files
    cred.parent.mkdir(parents=True, exist_ok=True)
    up = files.upload()
    cred.write_bytes(next(iter(up.values())))
    cred.chmod(0o600)
print('credentials ready')


## 3. ESC-50


In [ ]:
!python -m phase.data.download --dataset esc50 2>&1 | tail -4


## 4. Manifest, leakage-safe splits, feature cache

Grouped by source clip, so the several takes of one Freesound recording never
straddle a split.


In [ ]:
!python scripts/02_build_manifests.py --dataset esc50 2>&1 | head -10
!python scripts/03_build_splits.py --dataset esc50 2>&1 | grep -E 'train |val |test |wrote|SKIPPED' | head -6


In [ ]:
!python scripts/04_segment_and_cache.py --dataset esc50 --features logmel 2>&1 | grep -vE 'per class|per split|cached'


## 5. Confirm the split is leakage-free before training

If this is not zero, stop.


In [ ]:
import json

index = json.load(open('data/cache/esc50_fold_0_windows.json'))
groups = {}
for w in index['windows']:
    groups.setdefault(w['recording_id'], set()).add(w['split'])
straddling = sum(1 for v in groups.values() if len(v) > 1)
print('windows', index['n_windows'], ' recordings', len(groups), ' straddling', straddling)
assert straddling == 0, 'leakage detected, do not train'


## 6. Train, three seeds

`configs/eval/finetune.yaml` drives every hyperparameter - 40 epochs, batch 128, SGD
with cosine decay and warmup. Nothing is hardcoded here.

On a T4 this is roughly 15-25 minutes for all three seeds.


In [ ]:
import time
start = time.time()

!python scripts/06_evaluate.py --config configs/eval/finetune.yaml --cache data/cache/esc50_fold_0_logmel --seeds 0 1 2 --wandb-mode offline 2>&1 | grep -vE '^wandb|artifact'

print()
print('elapsed %.1f min' % ((time.time() - start) / 60))


## 7. Verdict


In [ ]:
import glob, json

path = sorted(glob.glob('results/tables/*seeds.json'))[-1]
payload = json.load(open(path))
summary = payload['summary']

print(path)
print('seeds', payload['seeds'])
print()
for key, stats in summary.items():
    print('%-12s %.4f +/- %.4f  (n=%d)' % (key, stats['mean'], stats['std'], stats['n']))

acc = summary['accuracy']['mean']
print()
if acc >= 0.55:
    print('PASS - %.1f%% on 50 classes. Phase 3 exit criterion met; Phase 6 can start.' % (100*acc))
elif acc >= 0.20:
    print('MARGINAL - %.1f%%, well above the 2%% chance level so the pipeline works,' % (100*acc))
    print('but below the expected 60-80%. The recipe underfits: investigate LR and epochs')
    print('before Phase 6, since the same trainer carries the linear probe and fine-tune.')
else:
    print('FAIL - %.1f%% is near the 2%% chance level. Real trainer bug;' % (100*acc))
    print('fix before Phase 6 rather than during pretraining.')


## 8. Augmentation cost on a real GPU

Closes D-020. The physics pipeline measures 142-166 ms per view-window on CPU and
33 ms on an MX450 - a 4.4x gain, which suggests it is memory-bandwidth bound, so a
T4 should be considerably faster.

> This needs commit `ba98799`, which fixes random sampling on non-CPU devices and is
> **not yet pushed**. If it reports a generator error, that is why. The ESC-50 gate
> above is unaffected - it never touches the augmentations.


In [ ]:
import time, torch
from phase.config import AugmentConfig, load_config
from phase.augment import Pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipeline = Pipeline(load_config('configs/augment/physics.yaml', AugmentConfig))
generator = torch.Generator(device='cpu').manual_seed(0)

try:
    for batch_size in (8, 32, 128):
        x = torch.randn(batch_size, 32000 * 30, device=device)
        pipeline.views(x, 32000, generator)
        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(3):
            pipeline.views(x, 32000, generator)
        if device == 'cuda':
            torch.cuda.synchronize()
        elapsed = (time.time() - start) / 3
        print('B=%3d  %6.2f s/batch  %6.1f ms per view-window'
              % (batch_size, elapsed, 1000 * elapsed / (2 * batch_size)))
        del x
        if device == 'cuda':
            torch.cuda.empty_cache()
    print()
    print('Keep s/batch well under ~2 s or 200 epochs of Phase 6 will not finish.')
except RuntimeError as exc:
    print('benchmark skipped:', exc)
    print()
    print('That is the unpushed generator-device fix (commit ba98799).')
    print('The ESC-50 gate above is unaffected.')


## 9. Download the result

Send me this file and I will record it in `docs/results_log.md` and close the gate.


In [ ]:
from google.colab import files
files.download(path)
